In [1]:
# Transilien BI Project

## Objective
"""
Analyze train punctuality and identify influencing factors such as:
- weather
- strikes
- holidays
- temporal patterns
"""

## Pipeline
"""
Raw data → Cleaning → Enrichment → Export for SAP Analytics Cloud
"""

'\nRaw data → Cleaning → Enrichment → Export for SAP Analytics Cloud\n'

In [2]:
## Import libraries
"""
We import the libraries required for:
- data manipulation
- feature engineering
- holiday generation
"""
import pandas as pd
import holidays
import matplotlib.pyplot as plt
import numpy as np

In [3]:
## Load the Transilien dataset
"""
The original dataset contains monthly punctuality indicators for Transilien lines.
"""
df = pd.read_csv("../Data/processed/ponctualite-mensuelle-transilien.csv",sep=";")
df.head()

,Date,Service,Ligne,Nom de la ligne,Taux de ponctualité,Nombre de voyageurs à l'heure pour un voyageur en retard
0,2013-01,RER,A,RER A,83.6,5.1
1,2013-01,Transilien,R,Paris Sud Est,87.2,6.8
2,2013-03,Transilien,H,Paris Nord Ouest,92.3,12.0
3,2013-04,Transilien,N,Paris Montparnasse,90.2,9.2
4,2013-05,RER,D,RER D,87.1,6.8


In [4]:
## Data cleaning
"""
We standardize column names and convert data types to prepare the dataset for analysis.
"""
df.columns =(
    df.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
    .str.replace("'","_")
    .str.replace("é","e")

    )
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2009 entries, 0 to 2008
Data columns (total 6 columns):
 #   Column                                                    Non-Null Count  Dtype  
---  ------                                                    --------------  -----  
 0   date                                                      2009 non-null   str    
 1   service                                                   2009 non-null   str    
 2   ligne                                                     2009 non-null   str    
 3   nom_de_la_ligne                                           2009 non-null   str    
 4   taux_de_ponctualite                                       2008 non-null   float64
 5   nombre_de_voyageurs_à_l_heure_pour_un_voyageur_en_retard  2009 non-null   float64
dtypes: float64(2), str(4)
memory usage: 94.3 KB


In [5]:
""" à ajouter en anglais"""
df["date"]=pd.to_datetime(df["date"])
df = df.dropna(subset="date")
df.info()



<class 'pandas.DataFrame'>
RangeIndex: 2009 entries, 0 to 2008
Data columns (total 6 columns):
 #   Column                                                    Non-Null Count  Dtype         
---  ------                                                    --------------  -----         
 0   date                                                      2009 non-null   datetime64[us]
 1   service                                                   2009 non-null   str           
 2   ligne                                                     2009 non-null   str           
 3   nom_de_la_ligne                                           2009 non-null   str           
 4   taux_de_ponctualite                                       2008 non-null   float64       
 5   nombre_de_voyageurs_à_l_heure_pour_un_voyageur_en_retard  2009 non-null   float64       
dtypes: datetime64[us](1), float64(2), str(3)
memory usage: 94.3 KB


In [6]:
## Feature engineering
"""
We create temporal variables and KPIs to improve BI analysis.
"""
df["annee"]=df["date"].dt.year
df["mois"]=df["date"].dt.month
df["nom_mois"]=df["date"].dt.month_name()
df["trimestre"]=df["date"].dt.quarter
df["taux_irregularite"] = 100 - df["taux_de_ponctualite"]

df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 2009 entries, 0 to 2008
Data columns (total 11 columns):
 #   Column                                                    Non-Null Count  Dtype         
---  ------                                                    --------------  -----         
 0   date                                                      2009 non-null   datetime64[us]
 1   service                                                   2009 non-null   str           
 2   ligne                                                     2009 non-null   str           
 3   nom_de_la_ligne                                           2009 non-null   str           
 4   taux_de_ponctualite                                       2008 non-null   float64       
 5   nombre_de_voyageurs_à_l_heure_pour_un_voyageur_en_retard  2009 non-null   float64       
 6   annee                                                     2009 non-null   int32         
 7   mois                                                 

,date,service,ligne,nom_de_la_ligne,taux_de_ponctualite,nombre_de_voyageurs_à_l_heure_pour_un_voyageur_en_retard,annee,mois,nom_mois,trimestre,taux_irregularite
0,2013-01-01,RER,A,RER A,83.6,5.1,2013,1,January,1,16.4
1,2013-01-01,Transilien,R,Paris Sud Est,87.2,6.8,2013,1,January,1,12.8
2,2013-03-01,Transilien,H,Paris Nord Ouest,92.3,12.0,2013,3,March,1,7.7
3,2013-04-01,Transilien,N,Paris Montparnasse,90.2,9.2,2013,4,April,2,9.8
4,2013-05-01,RER,D,RER D,87.1,6.8,2013,5,May,2,12.9


In [10]:
## Holiday enrichment
"""
French holidays are generated dynamically using the holidays Python package.
"""
fr_holidays = holidays.France(years=df["annee"].unique()) 
"""
We create a function that calculates the number of French holidays
for a month passed as a parameter.
"""
def count_holidays_in_month(date):
    #first day of the month
    start = date.replace(day=1)
    #last day of the month using MonthEnd offset
    end = start + pd.offsets.MonthEnd(0)
    #Generate all days in the month
    days = pd.date_range(start, end, freq="D")
    #Count how many of these days are in the list of French holidays
    return sum(day.date() in fr_holidays for day in days)




In [11]:
"""We apply the function to the date column 
to create a new variable that counts the number
 of holidays in each month."""

df["nb_jours_feries_mois"] = df["date"].apply(count_holidays_in_month)
df.head()

,date,service,ligne,nom_de_la_ligne,taux_de_ponctualite,nombre_de_voyageurs_à_l_heure_pour_un_voyageur_en_retard,annee,mois,nom_mois,trimestre,taux_irregularite,nb_jours_feries,nb_jours_feries_mois
0,2013-01-01,RER,A,RER A,83.6,5.1,2013,1,January,1,16.4,1,1
1,2013-01-01,Transilien,R,Paris Sud Est,87.2,6.8,2013,1,January,1,12.8,1,1
2,2013-03-01,Transilien,H,Paris Nord Ouest,92.3,12.0,2013,3,March,1,7.7,0,0
3,2013-04-01,Transilien,N,Paris Montparnasse,90.2,9.2,2013,4,April,2,9.8,1,1
4,2013-05-01,RER,D,RER D,87.1,6.8,2013,5,May,2,12.9,4,4


In [13]:
## Validate holiday feature consistency

# Check that the new feature was created
print(df.columns)

# Display holiday counts for several months
print(df[
    ["date", "nom_mois", "nb_jours_feries_mois"]
].head(20))

# Verify there are no missing values
print(df["nb_jours_feries_mois"].isna().sum())

# Analyze holiday count distribution
df["nb_jours_feries_mois"].value_counts()

Index(['date', 'service', 'ligne', 'nom_de_la_ligne', 'taux_de_ponctualite',
       'nombre_de_voyageurs_à_l_heure_pour_un_voyageur_en_retard', 'annee',
       'mois', 'nom_mois', 'trimestre', 'taux_irregularite', 'nb_jours_feries',
       'nb_jours_feries_mois'],
      dtype='str')
         date   nom_mois  nb_jours_feries_mois
0  2013-01-01    January                     1
1  2013-01-01    January                     1
2  2013-03-01      March                     0
3  2013-04-01      April                     1
4  2013-05-01        May                     4
5  2013-05-01        May                     4
6  2013-06-01       June                     0
7  2013-06-01       June                     0
8  2013-08-01     August                     1
9  2013-09-01  September                     0
10 2013-09-01  September                     0
11 2013-10-01    October                     0
12 2013-10-01    October                     0
13 2013-11-01   November                     2
14 2013-11-

nb_jours_feries_mois
1    894
0    791
2    169
4     91
3     64
Name: count, dtype: int64

In [14]:
## Seasonal feature engineering
"""
We create a function to determine the season
based on the month extracted from the date.
"""
def get_season(month):
    if month in [12, 1, 2]:
        return "Hiver"
    elif month in [3, 4, 5]:
        return "Printemps"
    elif month in [6, 7, 8]:
        return "Été"
    else:
        return "Automne"


In [15]:
"""
We create a seasonal variable to improve
temporal and weather-related analysis.
"""
df["saison"] = df["mois"].apply(get_season)

In [19]:
## Feature validation

# Display unique month-season combinations
# to verify the season mapping
print(
    df[
        ["mois", "nom_mois", "saison"]
    ]
    .drop_duplicates()
    .sort_values("mois")
)

# Analyze season distribution
print(
    df["saison"]
    .value_counts()
)

# Verify there are no missing values
print(
    df["saison"]
    .isna()
    .sum()
)

# Display the first rows of the dataset
# to confirm the new feature
df[
    ["date", "nom_mois", "saison"]
].head(20)

    mois   nom_mois     saison
0      1    January      Hiver
78     2   February      Hiver
2      3      March  Printemps
3      4      April  Printemps
4      5        May  Printemps
6      6       June        Été
33     7       July        Été
8      8     August        Été
9      9  September    Automne
11    10    October    Automne
13    11   November    Automne
17    12   December      Hiver
saison
Automne      507
Été          506
Hiver        505
Printemps    491
Name: count, dtype: int64
0


,date,nom_mois,saison
0,2013-01-01,January,Hiver
1,2013-01-01,January,Hiver
2,2013-03-01,March,Printemps
3,2013-04-01,April,Printemps
4,2013-05-01,May,Printemps
5,2013-05-01,May,Printemps
6,2013-06-01,June,Été
7,2013-06-01,June,Été
8,2013-08-01,August,Été
9,2013-09-01,September,Automne
